In [5]:
import xml.etree.ElementTree as ET
import pandas as pd
import os

## Test Code

In [6]:
test_file = "../data/schedules/unitrans_route_A.xml"

In [7]:
tree = ET.parse(test_file)
root = tree.getroot()

stop_map = {stop.get("tag"): stop.text for stop in root.findall(".//header/stop")}

patch = {
    "22273": "El Cemonte Ave & Glide Drive (SB)",
    "22011": "Alhambra Drive & Mace Blvd (WB)",
    "22073": "5th St & Cantrill/Greystone (WB)",
    "22066": "H St & 2nd St / Amtrak (SB)",
    "22062_ar": "Memorial Union & Main Island (SB)"
}
stop_map.update(patch)
rows = []
for route in root.findall(".//route"):   
    route_tag = route.get("tag")
    service_class = route.get("serviceClass")
    direction = route.get("direction")

    for tr in route.findall(".//tr"):
        block_id = tr.get("blockID")
        for stop in tr.findall("stop"):
                tag = stop.get("tag")
                rows.append({
                    "route_tag": route_tag,
                    "service_class": service_class,
                    "direction": direction,
                    "block_id": block_id,
                    "stopTag": tag,
                    "stop_name": stop_map.get(tag, f"Unknown ({tag})"),
                    "arrival_time": stop.text,
                    "epoch": stop.get("epochTime")
                })
df = pd.DataFrame(rows)
df['epoch'] = pd.to_numeric(df['epoch'])
df['arrival_dt'] = pd.to_datetime(df['epoch'], unit='ms', errors='coerce')
df = df[df['service_class'] != "20260318"]
print(df['service_class'].value_counts())


service_class
MoTuTh       160
Service03    160
Wednesday    160
Friday       140
Name: count, dtype: int64


In [8]:
df

,route_tag,service_class,direction,block_id,stopTag,stop_name,arrival_time,epoch,arrival_dt
160,A,Friday,Inbound to UCD Mu,29,22062,El Cemonte Ave & Glide Drive (SB),07:15:00,26100000,1970-01-01 07:15:00
161,A,Friday,Inbound to UCD Mu,29,22065,Alhambra Drive & Mace Blvd (WB),07:20:00,26400000,1970-01-01 07:20:00
162,A,Friday,Inbound to UCD Mu,29,22072,5th St & Cantrill/Greystone (WB),07:25:00,26700000,1970-01-01 07:25:00
163,A,Friday,Inbound to UCD Mu,29,22012,H St & 2nd St / Amtrak (SB),07:31:00,27060000,1970-01-01 07:31:00
164,A,Friday,Inbound to UCD Mu,29,22273_ar,Memorial Union & Main Island (SB),07:50:00,28200000,1970-01-01 07:50:00
...,...,...,...,...,...,...,...,...,...
775,A,Wednesday,Outbound to El Cemonte,37,22273,El Cemonte Ave & Glide Drive (SB),21:40:00,78000000,1970-01-01 21:40:00
776,A,Wednesday,Outbound to El Cemonte,37,22011,Alhambra Drive & Mace Blvd (WB),21:45:00,78300000,1970-01-01 21:45:00
777,A,Wednesday,Outbound to El Cemonte,37,22073,5th St & Cantrill/Greystone (WB),21:52:00,78720000,1970-01-01 21:52:00
778,A,Wednesday,Outbound to El Cemonte,37,22066,H St & 2nd St / Amtrak (SB),21:56:00,78960000,1970-01-01 21:56:00


## Schedule Parsing function

In [9]:
def parse_unitrans_schedule(file_path:str) -> pd.DataFrame:
    """
    Takes in the filepath and parses the xml file
    Returns a df of the formatted data
    """

    # initial setup
    tree = ET.parse(file_path)
    root = tree.getroot()

    # create a dict map for the tags to translate to the route names
    stop_map = {stop.get("tag"): stop.text for stop in root.findall(".//header/stop")}

    patch = {
        "22273": "El Cemonte Ave & Glide Drive (SB)",
        "22011": "Alhambra Drive & Mace Blvd (WB)",
        "22073": "5th St & Cantrill/Greystone (WB)",
        "22066": "H St & 2nd St / Amtrak (SB)",
        "22062_ar": "Memorial Union & Main Island (SB)"
    }

    # apply the dict map
    stop_map.update(patch)

    rows = []

    # loop through all the routes in the file
    for route in root.findall(".//route"):   

        # get the header attributes
        route_tag = route.get("tag")
        service_class = route.get("serviceClass")
        direction = route.get("direction").lower()

        # loop through all the individal blocks
        for tr in route.findall(".//tr"):
            block_id = tr.get("blockID")

            # for each stop in the block, parse the data
            for stop in tr.findall("stop"):
                tag = stop.get("tag")
                raw_time = stop.text

                # if the time is "--" or epoch is "-1", skip this stop
                if raw_time == "--" or stop.get("epochTime") == "-1":
                    continue

                # add attributes to the row list
                rows.append({
                    "route_tag": route_tag,
                    "service_class": service_class,
                    "direction": direction,
                    "block_id": block_id,
                    "stopTag": tag,
                    "stop_name": stop_map.get(tag, f"Unknown ({tag})"),
                    "arrival_time_str": raw_time,
                })

    # transform into df
    df = pd.DataFrame(rows)

    df['arrival_time'] = df['arrival_time_str'].str.strip()
    df['block_id'] = df['block_id'].astype(str)

    # only select the normal service routes
    df = df[df['service_class'].isin(["MoTuTh", 'Wednesday', 'Friday', 'Saturday', 'Sunday'])]

    return df


In [10]:
schedule_folder = "../data/schedules"

# initialize master df
master_df = pd.DataFrame()

# for each file, parse the data
for file in os.listdir(schedule_folder):
    # ensure it is xml (im looking at you curtis :eyes:)
    if not file.endswith(".xml") or file.startswith("."):
        continue
    
    filename = os.path.join(schedule_folder, file)
    current_df = parse_unitrans_schedule(filename)
    master_df = pd.concat([master_df, current_df], ignore_index=True)



In [11]:
master_df

,route_tag,service_class,direction,block_id,stopTag,stop_name,arrival_time_str,arrival_time
0,A,Friday,inbound to ucd mu,29,22062,El Cemonte Ave & Glide Drive (SB),07:15:00,07:15:00
1,A,Friday,inbound to ucd mu,29,22065,Alhambra Drive & Mace Blvd (WB),07:20:00,07:20:00
2,A,Friday,inbound to ucd mu,29,22072,5th St & Cantrill/Greystone (WB),07:25:00,07:25:00
3,A,Friday,inbound to ucd mu,29,22012,H St & 2nd St / Amtrak (SB),07:31:00,07:31:00
4,A,Friday,inbound to ucd mu,29,22273_ar,Memorial Union & Main Island (SB),07:50:00,07:50:00
...,...,...,...,...,...,...,...,...
14693,Z,Wednesday,outbound to target,6,22258,Silo Terminal & Haring Hall (WB),22:10:00,22:10:00
14694,Z,Wednesday,outbound to target,6,22011,Alhambra Drive & Mace Blvd (WB),22:17:00,22:17:00
14695,Z,Wednesday,outbound to target,6,22073,5th St & Cantrill/Greystone (WB),22:24:00,22:24:00
14696,Z,Wednesday,outbound to target,6,22066,H St & 2nd St / Amtrak (SB),22:28:00,22:28:00


In [12]:
import sqlite3

In [13]:
# create / connect to db
conn = sqlite3.connect('unitrans.db')

# create schedules table
master_df.to_sql('schedules', conn, if_exists='replace', index=False)

pd.read_sql_query("SELECT * FROM schedules", conn)

,route_tag,service_class,direction,block_id,stopTag,stop_name,arrival_time_str,arrival_time
0,A,Friday,inbound to ucd mu,29,22062,El Cemonte Ave & Glide Drive (SB),07:15:00,07:15:00
1,A,Friday,inbound to ucd mu,29,22065,Alhambra Drive & Mace Blvd (WB),07:20:00,07:20:00
2,A,Friday,inbound to ucd mu,29,22072,5th St & Cantrill/Greystone (WB),07:25:00,07:25:00
3,A,Friday,inbound to ucd mu,29,22012,H St & 2nd St / Amtrak (SB),07:31:00,07:31:00
4,A,Friday,inbound to ucd mu,29,22273_ar,Memorial Union & Main Island (SB),07:50:00,07:50:00
...,...,...,...,...,...,...,...,...
14693,Z,Wednesday,outbound to target,6,22258,Silo Terminal & Haring Hall (WB),22:10:00,22:10:00
14694,Z,Wednesday,outbound to target,6,22011,Alhambra Drive & Mace Blvd (WB),22:17:00,22:17:00
14695,Z,Wednesday,outbound to target,6,22073,5th St & Cantrill/Greystone (WB),22:24:00,22:24:00
14696,Z,Wednesday,outbound to target,6,22066,H St & 2nd St / Amtrak (SB),22:28:00,22:28:00
